# Проверка формулы индекса удовлетворённости

Проверяем формулу из контракта на двух моделях:

1. **Baseline:** TF-IDF по словам и символьным n-граммам + `LogisticRegression` из `sentiment_baseline.ipynb`.
2. **Выбранная DL-модель:** `r1char9/rubert-base-cased-russian-sentiment` из `sentiment_model_comparison.ipynb`.

Для каждого отзыва используем вероятности классов модели и считаем:

\[
I = 50 	imes \left(1 + \operatorname{mean}(p_{pos} - p_{neg})ight)
\]

Результат находится в диапазоне 0–100; 50 соответствует нулевому среднему перевесу вероятности позитива над негативом.


## Данные и сценарии

Используем размеченный test-split `mteb/RuReviewsClassification`. Из него детерминированно выбираем четыре выборки по 100 отзывов:

- 100 размеченных отрицательных;
- 100 размеченных положительных;
- 80 положительных + 20 отрицательных;
- 20 положительных + 80 отрицательных.

В выборках только крайние классы, чтобы проверить направление и линейность формулы. Метки используются для отбора отзывов, но индекс рассчитывается исключительно из `p_pos` и `p_neg` предсказаний каждой модели. При идеальных предсказаниях ожидаемые значения индекса — 0, 100, 80 и 20 соответственно; реальные значения могут отличаться из-за ошибок и уверенности моделей.


In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
from transformers import pipeline

DATASET_ID = "mteb/RuReviewsClassification"
SELECTED_MODEL_ID = "r1char9/rubert-base-cased-russian-sentiment"
CLASS_NAMES = ["negative", "neutral", "positive"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
SAMPLE_SIZE = 100
SAMPLE_SEED = 42
MAX_LENGTH = 512
BATCH_SIZE = 32
DEVICE = 0 if torch.cuda.is_available() else -1

# Загружаем train только для baseline, а test — только для оценки формулы.
dataset = load_dataset(DATASET_ID)
train_data = dataset["train"]
test_data = dataset["test"]
X_train, y_train = list(train_data["text"]), list(train_data["label"])
X_test, y_test = list(test_data["text"]), np.asarray(test_data["label"])

features = make_union(
    TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=3,
        sublinear_tf=True,
        max_features=300_000,
    ),
)
baseline = make_pipeline(
    features,
    LogisticRegression(C=4, max_iter=2000, class_weight="balanced"),
)
baseline.fit(X_train, y_train)

print(f"train: {len(X_train):,}; test: {len(X_test):,}")
print("test classes:", {name: int((y_test == i).sum()) for i, name in enumerate(CLASS_NAMES)})
print("baseline fitted")

train: 45,000; test: 2,048
test classes: {'negative': 682, 'neutral': 683, 'positive': 683}
baseline fitted


In [2]:
rng = np.random.default_rng(SAMPLE_SEED)
negative_ids = np.flatnonzero(y_test == LABEL_TO_ID["negative"])
positive_ids = np.flatnonzero(y_test == LABEL_TO_ID["positive"])

cohort_ids = {
    "100% negative": rng.choice(negative_ids, SAMPLE_SIZE, replace=False),
    "100% positive": rng.choice(positive_ids, SAMPLE_SIZE, replace=False),
    "80% positive / 20% negative": np.concatenate(
        [rng.choice(positive_ids, 80, replace=False), rng.choice(negative_ids, 20, replace=False)]
    ),
    "20% positive / 80% negative": np.concatenate(
        [rng.choice(positive_ids, 20, replace=False), rng.choice(negative_ids, 80, replace=False)]
    ),
}
for ids in cohort_ids.values():
    rng.shuffle(ids)

for name, ids in cohort_ids.items():
    counts = {
        "negative": int((y_test[ids] == LABEL_TO_ID["negative"]).sum()),
        "positive": int((y_test[ids] == LABEL_TO_ID["positive"]).sum()),
    }
    assert len(ids) == SAMPLE_SIZE
    print(f"{name}: n={len(ids)}, {counts}")

100% negative: n=100, {'negative': 100, 'positive': 0}
100% positive: n=100, {'negative': 0, 'positive': 100}
80% positive / 20% negative: n=100, {'negative': 20, 'positive': 80}
20% positive / 80% negative: n=100, {'negative': 80, 'positive': 20}


In [3]:
# Baseline возвращает вероятности в порядке baseline.classes_.
baseline_class_to_col = {label: i for i, label in enumerate(baseline.classes_)}

# DL-модель возвращает score каждого класса; считаем один раз для уникальных текстов.
all_test_ids = np.unique(np.concatenate(list(cohort_ids.values())))
selected_classifier = pipeline(
    "text-classification",
    model=SELECTED_MODEL_ID,
    device=DEVICE,
    top_k=None,
)
selected_outputs = selected_classifier(
    [X_test[i] for i in all_test_ids],
    batch_size=BATCH_SIZE,
    truncation=True,
    max_length=MAX_LENGTH,
)
selected_by_id = {
    int(i): {item["label"].lower(): item["score"] for item in output}
    for i, output in zip(all_test_ids, selected_outputs, strict=True)
}

results = []
for cohort_name, ids in cohort_ids.items():
    texts = [X_test[i] for i in ids]
    baseline_probs = baseline.predict_proba(texts)
    baseline_delta = np.mean(
        [
            row[baseline_class_to_col[LABEL_TO_ID["positive"]]]
            - row[baseline_class_to_col[LABEL_TO_ID["negative"]]]
            for row in baseline_probs
        ]
    )
    selected_delta = np.mean(
        [selected_by_id[int(i)]["positive"] - selected_by_id[int(i)]["negative"] for i in ids]
    )
    results.extend(
        [
            {
                "scenario": cohort_name,
                "model": "baseline TF-IDF + LogisticRegression",
                "mean_delta": baseline_delta,
                "satisfaction_index": 50 * (1 + baseline_delta),
            },
            {
                "scenario": cohort_name,
                "model": SELECTED_MODEL_ID,
                "mean_delta": selected_delta,
                "satisfaction_index": 50 * (1 + selected_delta),
            },
        ]
    )


def show_results(rows):
    print(f"{'Сценарий':31} {'Модель':48} {'mean(p_pos-p_neg)':>18} {'Индекс':>9}")
    for row in rows:
        print(
            f"{row['scenario']:31} {row['model']:48} "
            f"{row['mean_delta']:18.4f} {row['satisfaction_index']:9.2f}"
        )


show_results(results)

Сценарий                       Модель                                           mean(p_pos-p_neg)    Индекс
100% negative                   baseline TF-IDF + LogisticRegression                      -0.5723     21.39
100% negative                   r1char9/rubert-base-cased-russian-sentiment               -0.6092     19.54
100% positive                   baseline TF-IDF + LogisticRegression                       0.7884     89.42
100% positive                   r1char9/rubert-base-cased-russian-sentiment                0.8645     93.23
80% positive / 20% negative     baseline TF-IDF + LogisticRegression                       0.5169     75.85
80% positive / 20% negative     r1char9/rubert-base-cased-russian-sentiment                0.5123     75.62
20% positive / 80% negative     baseline TF-IDF + LogisticRegression                      -0.3661     31.70
20% positive / 80% negative     r1char9/rubert-base-cased-russian-sentiment               -0.4217     28.91


## Результат

Обе модели сохраняют ожидаемое направление индекса: полностью отрицательная выборка дает значение ниже 50, полностью положительная — выше 50; перевес 80/20 дает значение выше 50, перевес 20/80 — ниже 50. Значения не равны идеальным ориентирам (0, 100, 80 и 20), потому что формула агрегирует вероятности предсказаний, а модели ошибаются и по-разному калибруют уверенность.

| Сценарий | Baseline | Выбранная DL-модель | Идеальный ориентир |
|---|---:|---:|---:|
| 100% отрицательных | 21.39 | 19.54 | 0 |
| 100% положительных | 89.42 | 93.23 | 100 |
| 80% положительных / 20% отрицательных | 75.85 | 75.62 | 80 |
| 20% положительных / 80% отрицательных | 31.70 | 28.91 | 20 |

Тест подтверждает формулу как перевод среднего `p_pos − p_neg` в шкалу 0–100 с нейтральной точкой 50. Сравнение демонстрирует, что итог зависит от вероятностей конкретной модели; для сопоставимости формулы между версиями модели нужно сохранять `model_version` вместе с агрегатами.


In [4]:
indices = {
    row["scenario"]: row["satisfaction_index"]
    for row in results
    if row["model"] == SELECTED_MODEL_ID
}
assert indices["100% negative"] < 50
assert indices["100% positive"] > 50
assert indices["80% positive / 20% negative"] > 50
assert indices["20% positive / 80% negative"] < 50
print("Проверки направления индекса пройдены для выбранной DL-модели.")

Проверки направления индекса пройдены для выбранной DL-модели.
